# 4. PDF Loaders (PyPDFLoader & friends)

The loaders for **PDF files** — reports, research papers, invoices, books. PDFs are everywhere, so
this is one of the most-used loaders in real projects. "PDF loader" isn't a single class; it's a
**family**, and picking the right one matters.

---

## 1. Simple Definition

> **Kid version:** A PDF is like a **printed book with many pages**. `PyPDFLoader` is a helper that
> goes through the book **page by page**, reads the words on each page, and puts **each page in its
> own box** (`Document`) with a sticky note saying which page number it was.

**Professional definition:** `PyPDFLoader` reads a PDF and returns **one `Document` per page**, with
the page number stored in metadata. It uses the `pypdf` library under the hood.

```python
from langchain_community.document_loaders import PyPDFLoader

docs = PyPDFLoader("report.pdf").load()
print(len(docs))              # = number of pages
print(docs[0].page_content)   # text of page 1
print(docs[0].metadata)       # {"source": "report.pdf", "page": 0}
```

> **Install note:** `pip install pypdf`.

---

## 2. Why Does It Exist?

**The problem:** PDFs are a **binary** format designed for *printing*, not for reading by code. Text
can be in columns, tables, headers/footers, or even stored as **images** (scanned documents). You
can't just `open().read()` a PDF.

### Before LangChain

```python
import pypdf
reader = pypdf.PdfReader("report.pdf")
pages = [p.extract_text() for p in reader.pages]
# Then manually attach page numbers, build Documents, handle errors...
```

### After LangChain

```python
docs = PyPDFLoader("report.pdf").load()
# → one Document per page, with {"page": n} metadata, ready for the pipeline.
```

**One `Document` per page** is ideal: retrieval can return the exact page an answer came from, and
your citations can say "see page 12."

---

## 3. Real-Life Analogy

A **book scanner at a library** 📚. It flips through the book one page at a time, scans the text, and
files each page as a separate sheet stamped with its page number. Later you can pull out "page 12"
directly instead of re-reading the whole book.

---

## 4. Where It Fits in LangChain Architecture

```
BaseLoader
    │
    ▼
BasePDFLoader
    │
    ├── PyPDFLoader          ← fast, page-by-page (most common default)
    ├── PyMuPDFLoader        ← fast + rich metadata (uses "fitz"/PyMuPDF)
    ├── PDFPlumberLoader     ← better at tables/layout
    ├── PyPDFium2Loader      ← good quality, permissive license
    └── UnstructuredPDFLoader← powerful parsing; can do OCR for scanned PDFs
```

- **`BaseLoader` → PDF loaders:** all share `.load()`/`.lazy_load()`.
- There are **many** PDF loaders because PDFs vary wildly — plain text vs. tables vs. scanned images.
  You choose based on your document and needs.

### Which PDF loader should I use?

| Situation | Loader |
|-----------|--------|
| General purpose, fast, simple | **`PyPDFLoader`** (start here) |
| Need rich metadata / speed | `PyMuPDFLoader` |
| Lots of **tables** / precise layout | `PDFPlumberLoader` |
| **Scanned** PDFs (images of text) → needs OCR | `UnstructuredPDFLoader` (with OCR deps) |

---

## 5. Internal Working

```
  "report.pdf"  (binary)
        │
        ▼
  OPEN with pypdf  → detects N pages
        │
        ▼
  FOR EACH page:
     extract_text()  → "Chapter 1 ..."
        │
        ▼  wrap in a Document
     Document(page_content="Chapter 1 ...", metadata={"source": "report.pdf", "page": 0})
        │
        ▼
  return [ Document(page0), Document(page1), ... ]
```

> ⚠️ **Gotcha:** If a page returns **empty text**, the PDF is likely a **scanned image**. Regular
> extraction can't read images — you need an **OCR**-capable loader (e.g. `UnstructuredPDFLoader`
> with OCR, or a dedicated OCR step).

---

## 6. Attributes (constructor arguments)

### `file_path`

**Definition:** Path (or, for some loaders, URL) of the PDF.

In [ ]:
from pprint import pprint

def pretty_print_doc(doc):
    print("=" * 80)
    print("📄 CONTENT")
    print("-" * 80)
    print(doc.page_content)

    print("\n🏷️ METADATA")
    print("-" * 80)
    pprint(doc.metadata)

    print("=" * 80)
    print()

In [ ]:
from langchain_community.document_loaders.pdf import PyPDFLoader

loader = PyPDFLoader(file_path = r"knowledge-source\attention_is_all_you_need.pdf")
loader

In [ ]:
docs = loader.load()
pretty_print_doc(docs[0])

### `headers`

**Definition:** `headers` is a dictionary of **HTTP request headers** that `PyPDFLoader` sends **when downloading a PDF from a URL**. These headers are attached to the HTTP request before the PDF is fetched.

**Why it exists:** 

Many PDFs on the internet are **not publicly accessible**. They may require:

- Authentication (Bearer Token)
- API Key
- Session Cookie
- Custom User-Agent

Without these headers, the server may deny access (e.g., **401 Unauthorized** or **403 Forbidden**). The `headers` parameter lets you provide the required information so the PDF can be downloaded successfully.

**Real-life use case:** 

Suppose your company stores PDF reports on a secure server:

```text
https://company.com/reports/annual_report.pdf
```

Access is allowed only for logged-in employees.

You can load it by sending an authentication token:

The server verifies the token, allows the download, and `PyPDFLoader` extracts the PDF content.

**Important Notes:**

- ✅ Used **only** when `file_path` is a URL.
- ❌ Ignored when loading a local PDF:

- Common headers include:
  - `Authorization`
  - `x-api-key`
  - `Cookie`
  - `User-Agent`

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(file_path="https://company.com/reports/annual_report.pdf",
                     headers={"Authorization": "Bearer YOUR_ACCESS_TOKEN"})

docs = loader.load()

### `password`

**Definition:** Password to open an **encrypted/protected** PDF.

**Why it exists:** Many business PDFs (bank statements, invoices) are password-protected.

**When developers use it:** Loading secured PDFs.

**Real-life use case:** Unlocking a locked filing cabinet before scanning.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader(file_path="https://company.com/reports/annual_report.pdf",
                     password="my-secret")

docs = loader.load()

### `mode`

**Definition:** Some newer PDF loaders let you choose one `Document` **per page** (`"page"`) or one
`Document` for the **whole file** (`"single"`).

**Why it exists:** Sometimes you want the whole document as one blob (e.g. short PDFs), sometimes page
granularity (for citations).

In [6]:
from langchain_community.document_loaders.pdf import PyPDFLoader

loader = PyPDFLoader(file_path = r"knowledge-source\attention_is_all_you_need.pdf",
                     mode='single')

docs = loader.load()
print("Length of doc: ", len(docs))
pretty_print_doc(docs[0])

Length of doc:  1
📄 CONTENT
--------------------------------------------------------------------------------
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, the

In [7]:
from langchain_community.document_loaders.pdf import PyPDFLoader

loader = PyPDFLoader(file_path = r"knowledge-source\attention_is_all_you_need.pdf",
                     mode='page')

docs = loader.load()
print("Length of doc: ", len(docs))
pretty_print_doc(docs[0])

Length of doc:  15
📄 CONTENT
--------------------------------------------------------------------------------
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, th

### `extract_images`

**Definition:** If `True`, attempt to extract text **from images** inside the PDF (requires OCR
dependencies like `rapidocr-onnxruntime`).

**Why it exists:** Some text only exists as pictures; OCR recovers it.

**When developers use it:** PDFs with figures/screenshots containing important text.

**Real-life use case:** Reading the words printed inside a photo on the page.

In [8]:
from langchain_community.document_loaders.pdf import PyPDFLoader
from langchain_community.document_loaders.parsers import RapidOCRBlobParser

loader = PyPDFLoader(file_path = r"knowledge-source\attention_is_all_you_need.pdf",
                     mode='page',
                     extract_images=True,
                     images_parser=RapidOCRBlobParser(),)

docs = loader.load()
print("Length of doc: ", len(docs))
pretty_print_doc(docs[3])

[INFO] 2026-08-30 09:56:58,261 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-30 09:56:58,519 [RapidOCR] download_file.py:60: File exists and is valid: D:\Dev_Workspace\LangChain\langchainenv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-08-30 09:56:58,520 [RapidOCR] main.py:63: Using D:\Dev_Workspace\LangChain\langchainenv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-08-30 09:56:58,572 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-30 09:56:58,574 [RapidOCR] download_file.py:60: File exists and is valid: D:\Dev_Workspace\LangChain\langchainenv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-08-30 09:56:58,575 [RapidOCR] main.py:63: Using D:\Dev_Workspace\LangChain\langchainenv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-08-30 09:56:58,622 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-30 09:56:58,65

Length of doc:  15
📄 CONTENT
--------------------------------------------------------------------------------
Scaled Dot-Product Attention
 Multi-Head Attention
Figure 2: (left) Scaled Dot-Product Attention. (right) Multi-Head Attention consists of several
attention layers running in parallel.
of the values, where the weight assigned to each value is computed by a compatibility function of the
query with the corresponding key.
3.2.1 Scaled Dot-Product Attention
We call our particular attention "Scaled Dot-Product Attention" (Figure 2). The input consists of
queries and keys of dimension dk, and values of dimension dv. We compute the dot products of the
query with all keys, divide each by √dk, and apply a softmax function to obtain the weights on the
values.
In practice, we compute the attention function on a set of queries simultaneously, packed together
into a matrix Q. The keys and values are also packed together into matrices K and V . We compute
the matrix of outputs as:
Attention(

### `images_inner_format`

**Definition:**

Controls **how OCR-extracted text from images is inserted into the document's text** when `extract_images=True`.

After LangChain extracts text from images (using an OCR parser like `TesseractBlobParser` or `RapidOCRBlobParser`), it needs to decide **how that text should appear** in the final `Document.page_content`. `images_inner_format` lets you choose that representation.

**Why it exists:**

Many PDFs contain valuable information inside:

- Figures
- Diagrams
- Screenshots
- Tables saved as images
- Scanned pages

OCR converts these images into text, but LangChain still needs to decide **how to place that text** within the document.

`images_inner_format` defines **the format used to insert OCR text into the page content**.

**When developers use it:**

Developers use `images_inner_format` when they want to control how image-derived text is combined with the normal PDF text for:

- RAG (Retrieval-Augmented Generation)
- Semantic Search
- Embedding Generation
- Document Chunking
- HTML/Markdown document processing

**Real-life use case:**

Imagine you're reading a research paper.

The page contains:

- A paragraph explaining an experiment.
- A figure containing important labels and numbers.

Without OCR, the LLM only sees the paragraph.

With OCR enabled, the figure's text is extracted.

`images_inner_format` determines **how that extracted text is inserted into the page**, so the LLM can understand both the paragraph and the figure.

**How it works:**

Suppose a PDF page contains:

```text
Transformers have revolutionized NLP.

+-----------------------------+
| Self-Attention Mechanism    |
| Query • Key • Value         |
+-----------------------------+
```

The image contains:

```text
Self-Attention Mechanism
Query • Key • Value
```

OCR extracts:

```text
Self-Attention Mechanism
Query • Key • Value
```

Now `images_inner_format` decides how this OCR text appears inside the final document.

**Option 1: `"text"`**

**Description**

The OCR result is inserted as plain text.

Example output:

```text
Transformers have revolutionized NLP.

Self-Attention Mechanism
Query • Key • Value
```

**Best for**

- RAG
- Embeddings
- Semantic Search
- Question Answering
- Chunking

**Why choose it?**

Embedding models care about **text**, not formatting.

This is the recommended option for most LangChain applications.

---

**Option 2: `"markdown-img"`**

**Description**

The image is represented using Markdown, along with its OCR text.

Conceptually:

```markdown
Transformers have revolutionized NLP.

![Image](...)

Self-Attention Mechanism
Query • Key • Value
```

(The exact Markdown generated may vary depending on the loader implementation.)

**Best for**

- Markdown documents
- Documentation pipelines
- Preserving image context
- LLMs that benefit from Markdown structure

**Why choose it?**

The LLM knows that this content originally came from an image rather than normal text.

---

**Option 3: `"html-img"`**

**Description**

The image is represented using HTML.

Conceptually:

```html
<p>Transformers have revolutionized NLP.</p>

<img ...>

<p>
Self-Attention Mechanism
Query • Key • Value
</p>
```

(The exact HTML generated may vary depending on the loader implementation.)

**Best for**

- HTML documents
- Web applications
- Browser rendering
- HTML-based document pipelines

**Why choose it?**

Useful when your downstream application already works with HTML documents.

**Why this matters**

Suppose your PDF contains this figure:

```text
+------------------------+
| Accuracy = 98.7%       |
| Epochs = 50            |
+------------------------+
```

Without OCR:

```text
Figure 3
```

The important numbers are completely lost.

With OCR:

```text
Figure 3

Accuracy = 98.7%
Epochs = 50
```

Now an LLM can answer questions like:

> What accuracy did the model achieve?

because the text inside the image has been extracted.

In [9]:
from langchain_community.document_loaders.pdf import PyPDFLoader
from langchain_community.document_loaders.parsers import TesseractBlobParser, RapidOCRBlobParser

loader = PyPDFLoader(file_path = r"knowledge-source\attention_is_all_you_need.pdf",
                     mode='page',
                     extract_images=True,
                     images_parser=RapidOCRBlobParser(),
                     images_inner_format="text")

docs = loader.load()
print("Length of doc: ", len(docs))
pretty_print_doc(docs[0])

[INFO] 2026-08-30 09:57:03,820 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-30 09:57:03,827 [RapidOCR] download_file.py:60: File exists and is valid: D:\Dev_Workspace\LangChain\langchainenv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-08-30 09:57:03,828 [RapidOCR] main.py:63: Using D:\Dev_Workspace\LangChain\langchainenv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-08-30 09:57:03,872 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-30 09:57:03,873 [RapidOCR] download_file.py:60: File exists and is valid: D:\Dev_Workspace\LangChain\langchainenv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-08-30 09:57:03,874 [RapidOCR] main.py:63: Using D:\Dev_Workspace\LangChain\langchainenv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-08-30 09:57:03,920 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-30 09:57:03,93

Length of doc:  15
📄 CONTENT
--------------------------------------------------------------------------------
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, th

In [10]:
from langchain_community.document_loaders.pdf import PyPDFLoader
from langchain_community.document_loaders.parsers import TesseractBlobParser, RapidOCRBlobParser

loader = PyPDFLoader(file_path = r"knowledge-source\attention_is_all_you_need.pdf",
                     mode='page',
                     extract_images=True,
                     images_parser=RapidOCRBlobParser(),
                     images_inner_format="markdown-img")

docs = loader.load()
print("Length of doc: ", len(docs))
pretty_print_doc(docs[0])

[INFO] 2026-08-30 09:57:06,912 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-30 09:57:06,919 [RapidOCR] download_file.py:60: File exists and is valid: D:\Dev_Workspace\LangChain\langchainenv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-08-30 09:57:06,920 [RapidOCR] main.py:63: Using D:\Dev_Workspace\LangChain\langchainenv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-08-30 09:57:06,965 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-30 09:57:06,967 [RapidOCR] download_file.py:60: File exists and is valid: D:\Dev_Workspace\LangChain\langchainenv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-08-30 09:57:06,967 [RapidOCR] main.py:63: Using D:\Dev_Workspace\LangChain\langchainenv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-08-30 09:57:07,009 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-30 09:57:07,02

Length of doc:  15
📄 CONTENT
--------------------------------------------------------------------------------
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, th

In [11]:
from langchain_community.document_loaders.pdf import PyPDFLoader
from langchain_community.document_loaders.parsers import TesseractBlobParser, RapidOCRBlobParser

loader = PyPDFLoader(file_path = r"knowledge-source\attention_is_all_you_need.pdf",
                     mode='page',
                     extract_images=True,
                     images_parser=RapidOCRBlobParser(),
                     images_inner_format="html-img")

docs = loader.load()
print("Length of doc: ", len(docs))
pretty_print_doc(docs[0])

[INFO] 2026-08-30 09:57:09,975 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-30 09:57:09,982 [RapidOCR] download_file.py:60: File exists and is valid: D:\Dev_Workspace\LangChain\langchainenv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-08-30 09:57:09,982 [RapidOCR] main.py:63: Using D:\Dev_Workspace\LangChain\langchainenv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-08-30 09:57:10,022 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-30 09:57:10,023 [RapidOCR] download_file.py:60: File exists and is valid: D:\Dev_Workspace\LangChain\langchainenv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-08-30 09:57:10,024 [RapidOCR] main.py:63: Using D:\Dev_Workspace\LangChain\langchainenv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-08-30 09:57:10,069 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-30 09:57:10,08

Length of doc:  15
📄 CONTENT
--------------------------------------------------------------------------------
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, th

### `extraction_mode`

**Definition:**

`extraction_mode` specifies **which text extraction algorithm `PyPDFLoader` uses to read text from a PDF**.

Different PDFs store text differently. Some have simple paragraphs, while others contain multiple columns, tables, mathematical equations, or complex layouts.

The extraction mode tells the underlying PDF parser **how aggressively it should preserve the document's original layout**.

**Why it exists:**

Not all PDFs are created the same.

For example:

- Research papers often have **two-column layouts**.
- Financial reports contain **tables**.
- Books have **simple paragraphs**.
- Magazines contain **images mixed with text**.

A single extraction strategy cannot handle every document perfectly.

`extraction_mode` allows you to choose the most suitable extraction algorithm for your document.

**Available Values:**

Currently, `PyPDFLoader` supports two extraction modes:

- `"plain"` (default)
- `"layout"`

1. `"plain"` (Default)

**Definition:**

Extracts only the textual content while **ignoring most of the page's visual layout**.

The goal is to produce clean, continuous text.

---

**How it works:**

Suppose your PDF looks like this:

```text
--------------------------------------
Introduction

Transformers are neural networks
designed for sequence modeling.

Figure 1

Conclusion
--------------------------------------
```

Using:

```python
loader = PyPDFLoader(
    "paper.pdf",
    extraction_mode="plain"
)
```

The extracted text becomes:

```text
Introduction

Transformers are neural networks
designed for sequence modeling.

Figure 1

Conclusion
```

Notice that:

- Paragraphs are preserved.
- Formatting is simplified.
- Exact spacing and positioning are ignored.

---

**Best for**

- RAG
- Embeddings
- Semantic Search
- Question Answering
- LLM applications

---

**Advantages:**

- Clean output
- Smaller text
- Easier chunking
- Better embeddings

---

**Disadvantages:**

- Loses page layout
- Columns may merge
- Tables may become difficult to read

---

**Real-life use case**

Imagine copying text from a PDF into Notepad.

You don't care where every word appeared on the page—you just want the readable text.

That's essentially what `"plain"` mode does.

---

2. `"layout"`

**Definition:**

Attempts to preserve the **visual arrangement of the page**.

Instead of extracting only the text, it also tries to maintain:

- Spaces
- Line breaks
- Columns
- Relative positioning

---

**How it works**

Original PDF:

```text
Name          Salary

Alice         5000
Bob           6000
```

Using:

```python
loader = PyPDFLoader(
    "report.pdf",
    extraction_mode="layout"
)
```

The extracted text might look like:

```text
Name          Salary
Alice         5000
Bob           6000
```

Notice that the spacing is preserved, making the table easier to understand.

---

**Best for**

- Tables
- Forms
- Financial reports
- Scientific papers
- Multi-column documents

---

**Advantages:**

- Preserves document structure
- Better for tables
- Better for aligned text
- Better for OCR post-processing

---

**Disadvantages:**

- More whitespace
- Harder to chunk
- Less clean for embeddings
- May include extra blank lines

---

**Real-life use case**

Imagine taking a screenshot of a page and typing it exactly as it appears, including spacing and alignment.

That's what `"layout"` mode tries to achieve.

---

**Comparison Example:**

Original PDF:

```text
-----------------------------------------
Product       Price

Laptop        $1200
Mouse         $40

Total         $1240
-----------------------------------------
```

##### `"plain"`

```text
Product
Price
Laptop
1200
Mouse
40
Total
1240
```

The information is present, but the table structure is mostly lost.

---

##### `"layout"`

```text
Product       Price

Laptop        $1200
Mouse         $40

Total         $1240
```

The alignment is preserved, making it much easier for both humans and LLMs to interpret the table.

---

##### When to use each mode

| Extraction Mode | Preserves Layout | Best For |
|-----------------|------------------|----------|
| `"plain"` | ❌ No | RAG, embeddings, semantic search, QA |
| `"layout"` | ✅ Yes | Tables, forms, financial reports, multi-column PDFs |

---

**Which one should you use?**

##### Use `"plain"` if:

- You're building a RAG application.
- You're creating embeddings.
- You want clean, continuous text.
- Your PDFs mostly contain paragraphs.

Example:

```python
loader = PyPDFLoader(
    "paper.pdf",
    extraction_mode="plain"
)
```

---

##### Use `"layout"` if:

- The PDF contains tables.
- The document has multiple columns.
- Relative spacing is important.
- You need to preserve the visual structure.

Example:

```python
loader = PyPDFLoader(
    "financial_report.pdf",
    extraction_mode="layout"
)
```

In [12]:
from langchain_community.document_loaders.pdf import PyPDFLoader
from langchain_community.document_loaders.parsers import TesseractBlobParser, RapidOCRBlobParser

loader = PyPDFLoader(file_path = r"knowledge-source\attention_is_all_you_need.pdf",
                     mode='page',
                     extract_images=True,
                     images_parser=RapidOCRBlobParser(),
                     images_inner_format="html-img",
                     extraction_mode = "plain",
                    )

docs = loader.load()
print("Length of doc: ", len(docs))
pretty_print_doc(docs[0])

[INFO] 2026-08-30 09:57:14,759 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-30 09:57:14,766 [RapidOCR] download_file.py:60: File exists and is valid: D:\Dev_Workspace\LangChain\langchainenv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-08-30 09:57:14,766 [RapidOCR] main.py:63: Using D:\Dev_Workspace\LangChain\langchainenv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-08-30 09:57:14,810 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-30 09:57:14,812 [RapidOCR] download_file.py:60: File exists and is valid: D:\Dev_Workspace\LangChain\langchainenv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-08-30 09:57:14,812 [RapidOCR] main.py:63: Using D:\Dev_Workspace\LangChain\langchainenv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-08-30 09:57:14,858 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-30 09:57:14,87

Length of doc:  15
📄 CONTENT
--------------------------------------------------------------------------------
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗ †
University of Toronto
aidan@cs.toronto.edu
Łukasz Kaiser∗
Google Brain
lukaszkaiser@google.com
Illia Polosukhin∗ ‡
illia.polosukhin@gmail.com
Abstract
The dominant sequence transduction models are based on complex recurrent or
convolutional neural networks that include an encoder and a decoder. The best
performing models also connect the encoder and decoder through an attention
mechanism. We propose a new simple network architecture, th

In [13]:
from langchain_community.document_loaders.pdf import PyPDFLoader
from langchain_community.document_loaders.parsers import TesseractBlobParser, RapidOCRBlobParser

loader = PyPDFLoader(file_path = r"knowledge-source\attention_is_all_you_need.pdf",
                     mode='page',
                     extract_images=True,
                     images_parser=RapidOCRBlobParser(),
                     images_inner_format="html-img",
                     extraction_mode = "layout",
                    )

docs = loader.load()
print("Length of doc: ", len(docs))
pretty_print_doc(docs[0])

Rotated text discovered. Output will be incomplete.
[INFO] 2026-08-30 09:57:17,931 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-30 09:57:17,938 [RapidOCR] download_file.py:60: File exists and is valid: D:\Dev_Workspace\LangChain\langchainenv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-08-30 09:57:17,938 [RapidOCR] main.py:63: Using D:\Dev_Workspace\LangChain\langchainenv\Lib\site-packages\rapidocr\models\PP-OCRv6_det_small.onnx
[INFO] 2026-08-30 09:57:17,979 [RapidOCR] base.py:23: Using engine_name: onnxruntime
[INFO] 2026-08-30 09:57:17,981 [RapidOCR] download_file.py:60: File exists and is valid: D:\Dev_Workspace\LangChain\langchainenv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-08-30 09:57:17,981 [RapidOCR] main.py:63: Using D:\Dev_Workspace\LangChain\langchainenv\Lib\site-packages\rapidocr\models\ch_ppocr_mobile_v2.0_cls_mobile.onnx
[INFO] 2026-08-30 09:57:18,026 [RapidOCR] base.py:23: Using en

Length of doc:  15
📄 CONTENT
--------------------------------------------------------------------------------
Provided proper attribution is provided, Google hereby grants permission to
    reproduce the tables and figures in this paper solely for use in journalistic or
                                              scholarly works.


                              Attention Is All You Need







       Ashish Vaswani∗                Noam Shazeer∗                Niki Parmar∗             Jakob Uszkoreit∗
         Google Brain                  Google Brain              Google Research            Google Research
  avaswani@google.com              noam@google.com             nikip@google.com            usz@google.com

          Llion Jones∗                   Aidan N. Gomez∗ †                         Łukasz Kaiser∗
       Google Research                 University of Toronto                        Google Brain
     llion@google.com                aidan@cs.toronto.edu                  lukaszk